In [1]:
%%time
%matplotlib inline

import importlib
import new_import_ODC  

importlib.reload(new_import_ODC)

from new_import_ODC import *

print("✅ All modules loaded successfully")

✅ All modules loaded successfully
CPU times: user 12.8 s, sys: 1.7 s, total: 14.4 s
Wall time: 15.6 s


In [ ]:
%%time
import os
import sys

print("✅ AWS credentials loaded from environment variables")

# Cấu hình Dask local
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4)
client = Client(cluster)
print("✅ Dask cluster initialized")
print(f"   Cluster: {cluster}")

# Khai báo Datacube (chỉ để lấy metadata, không dùng load())
import datacube
try:
    dc = datacube.Datacube()
    print("✅ Datacube connected (metadata only)")
except Exception as e:
    print(f"⚠️  Datacube connection not critical: {e}")
    dc = None

print("\n" + "="*70)

✅ AWS credentials loaded from environment variables
✅ Dask cluster initialized
   Cluster: LocalCluster(6d2dfebe, 'tcp://127.0.0.1:35359', workers=4, threads=24, memory=31.26 GiB)
✅ Datacube connected (metadata only)

CPU times: user 631 ms, sys: 213 ms, total: 845 ms
Wall time: 1.86 s


2025-11-05 02:14:47,067 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:36317'.
2025-11-05 02:14:47,072 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:42289'.
2025-11-05 02:14:47,074 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:44491'.
2025-11-05 02:14:47,077 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:33673'.


In [3]:
%%time
# 🔧 Get Sentinel-2 scene metadata from datacube
print("="*70)
print("GETTING SENTINEL-2 SCENE METADATA")
print("="*70)

date_range = ("2023-03-01", "2023-12-31")
longtitude_range = (105.5, 106.4)
latitude_range = (9.2, 10.0)

try:
    print(f"\n[1] Loading metadata from datacube...")
    datasets = list(dc.find_datasets(product='s2_l2a', time=date_range))
    print(f"    ✅ Found {len(datasets)} scenes")
    
    if datasets:
        selected = datasets[0]
        print(f"\n[2] Selected scene: {selected.metadata.label}")
        scene_datetime = selected.time.begin if hasattr(selected.time, 'begin') else selected.time
        print(f"    Date: {scene_datetime}")
        
        # Display measurement paths
        print(f"\n[3] Available bands:")
        for name, measurement in selected.measurements.items():
            print(f"    - {name}: {measurement['path'][:80]}")
            
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print("="*70)

GETTING SENTINEL-2 SCENE METADATA

[1] Loading metadata from datacube...
    ✅ Found 40 scenes

[2] Selected scene: S2A_48PWR_20231226_0_L2A
    Date: 2023-12-26 03:35:26.919000+00:00

[3] Available bands:
    - nir: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - red: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - scl: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - blue: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - green: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - nir08: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - nir09: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - swir16: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/48/P/WR/20
    - swir22: https://sentinel-cogs.s3.us-west-2

In [4]:
%%time
# 🔍 CHECK IF DATASET CACHE EXISTS (Skip download if available)
print("="*70)
print("CHECKING FOR CACHED DATASET")
print("="*70)

import os
import xarray as xr

cache_dir = "dataset_cache"
cache_file = f"{cache_dir}/sentinel2_timeseries_40scenes.nc"

use_cache = False

if os.path.exists(cache_file):
    print(f"\n✅ Cache file found: {cache_file}")
    
    # Get file info
    file_size_gb = os.path.getsize(cache_file) / (1024**3)
    print(f"   File size: {file_size_gb:.2f} GB")
    
    # Try to load
    try:
        print(f"\n🔄 Loading dataset from cache...")
        data = xr.open_dataset(cache_file)
        
        print(f"✅ Dataset loaded from cache!")
        print(f"   Total scenes: {len(data['time'])}")
        print(f"   Variables: {len(data.data_vars)}")
        print(f"   Dimensions: {dict(data.dims)}")
        print(f"\n   ⏭️  Skipping S3 download (using cached data)")
        
        use_cache = True
        
    except Exception as e:
        print(f"❌ Error loading cache: {e}")
        print(f"   Will download fresh data from S3")
        use_cache = False
else:
    print(f"\n⏳ Cache file not found: {cache_file}")
    print(f"   Will download from S3 and save cache")
    print(f"   (Next run will use cache automatically)")

print("="*70)

CHECKING FOR CACHED DATASET

⏳ Cache file not found: dataset_cache/sentinel2_timeseries_40scenes.nc
   Will download from S3 and save cache
   (Next run will use cache automatically)
CPU times: user 343 μs, sys: 52 μs, total: 395 μs
Wall time: 359 μs


In [ ]:
%%time
# 💾 LOAD SENTINEL-2 DATA DIRECTLY FROM S3 COGS USING RASTERIO - WITH TEMPORAL FEATURES
print("="*70)
print("LOADING SENTINEL-2 DATA FROM S3 COGs (RASTERIO) - OPTIMAL ACCURACY")
print("="*70)

try:
    import rasterio
    import xarray as xr
    import numpy as np
    from scipy import ndimage
    
    # ===== CHECK IF SHOULD SKIP DOWNLOAD =====
    if use_cache and data is not None:
        print(f"\n✅ Using cached dataset - skipping download!")
        print(f"   Variables: {len(data.data_vars)}")
        print(f"   Shape: {data.dims}")
        display(data)
    
    else:
        # ===== DOWNLOAD FROM S3 =====
        print(f"\n📥 Downloading from S3...")
        
        # Get all scenes from datacube metadata
        datasets = list(dc.find_datasets(
            product='s2_l2a',
            time=date_range
        ))
        
        if not datasets:
            raise ValueError("No datasets found for date range")
        
        print(f"\n📦 Found {len(datasets)} available scenes")
        print(f"   Date range: {date_range[0]} to {date_range[1]}")
        
        # ===== LOAD ALL SCENES WITH ALL AVAILABLE BANDS (NO MAGNIFICATION) =====
        print(f"\n[LOADING] Loading ALL {len(datasets)} scenes with ALL available bands...")
        print(f"   (Keeping NATIVE resolution - NO upsampling/magnification)")
        
        # num_scenes = len(datasets)  # Load ALL scenes
        num_scenes = 1 # Load ALL scenes
        all_data_dict = {}
        failed_scenes = []
        scene_dates = []
        
        # Discover all available bands from first scene
        first_scene = datasets[0]
        all_available_bands = list(first_scene.measurements.keys())
        print(f"   Available bands: {all_available_bands}")
        
        for scene_idx in range(num_scenes):
            selected = datasets[scene_idx]
            scene_label = selected.metadata.label
            scene_datetime = selected.time.begin if hasattr(selected.time, 'begin') else selected.time
            scene_dates.append(scene_datetime)
            
            # Print progress every 5 scenes
            if scene_idx % 5 == 0 or scene_idx == 0 or scene_idx == num_scenes - 1:
                print(f"\n   [{scene_idx + 1:2d}/{num_scenes}] {scene_label} ({scene_datetime.date()})")
            
            # Load ALL available bands from S3 COGs
            scene_data_dict = {}
            
            for band_name in all_available_bands:
                if band_name in selected.measurements:
                    band_path = selected.measurements[band_name]['path']
                    
                    try:
                        with rasterio.open(band_path) as src:
                            data_band = src.read(1)
                            scene_data_dict[band_name] = data_band
                    except Exception as e:
                        if scene_idx % 5 == 0:
                            print(f"      ⚠️ Error loading {band_name}: {str(e)[:30]}")
                        failed_scenes.append((scene_idx, scene_label, band_name, str(e)))
            
            if scene_data_dict:
                all_data_dict[scene_idx] = scene_data_dict
                if scene_idx % 5 == 0 or scene_idx == num_scenes - 1:
                    print(f"      ✅ {len(scene_data_dict)} bands loaded")
            else:
                failed_scenes.append((scene_idx, scene_label, "all", "No bands loaded"))
        
        if not all_data_dict:
            raise ValueError("Could not load any bands from any scene")
        
        print(f"\n✅ Successfully loaded {len(all_data_dict)} scenes!")
        if failed_scenes:
            print(f"⚠️  Failed to load {len(failed_scenes)} band instances (will be skipped)")
        
        # ===== NORMALIZE RESOLUTION (No upsampling - just match to highest) =====
        print(f"\n[RESOLUTION NORMALIZATION] Aligning all bands to native resolution (NO magnification)...")
        
        # Find max resolution
        ref_resolution = None
        max_size = 0
        max_band = None
        
        for scene_idx in all_data_dict.keys():
            for band_name, data_band in all_data_dict[scene_idx].items():
                size = data_band.shape[0]
                if size > max_size:
                    max_size = size
                    ref_resolution = size
                    max_band = band_name
        
        print(f"   Reference resolution: {max_size}×{max_size} pixels (native {max_band})")
        
        # Resample all bands to match reference resolution (both up and down)
        resampled_count = 0
        for scene_idx in all_data_dict.keys():
            for band_name in list(all_data_dict[scene_idx].keys()):
                band_data_arr = all_data_dict[scene_idx][band_name]
                current_size = band_data_arr.shape[0]
                
                if current_size != ref_resolution:
                    scale_factor = ref_resolution / current_size
                    
                    # Resample to match reference resolution (both up and down)
                    if band_name == 'scl':
                        resampled_data = ndimage.zoom(band_data_arr, scale_factor, order=0)
                    else:
                        resampled_data = ndimage.zoom(band_data_arr, scale_factor, order=1)
                    
                    all_data_dict[scene_idx][band_name] = resampled_data
                    new_size = resampled_data.shape[0]
                    if scene_idx == 0:  # Print for first scene only to reduce clutter
                        print(f"      Resampling {band_name}: {current_size}×{current_size} → {new_size}×{new_size}")
                    resampled_count += 1
        
        print(f"✅ Resolution normalization complete! ({resampled_count} bands resampled)")
        
        # ===== CALCULATE SPECTRAL INDICES FOR EACH SCENE =====
        print(f"\n[SPECTRAL INDICES] Calculating spectral indices for each scene...")
        
        indices_count = 0
        for scene_idx in all_data_dict.keys():
            scene_data = all_data_dict[scene_idx]
            
            try:
                # NDVI: (NIR - Red) / (NIR + Red)
                if 'nir' in scene_data and 'red' in scene_data:
                    nir = scene_data['nir'].astype(float)
                    red = scene_data['red'].astype(float)
                    ndvi = (nir - red) / (nir + red + 1e-8)
                    scene_data['ndvi'] = ndvi.astype(np.float32)
                    indices_count += 1
                
                # NDBI: (SWIR - NIR) / (SWIR + NIR)
                if 'b11' in scene_data and 'nir' in scene_data:
                    swir = scene_data['b11'].astype(float)
                    nir = scene_data['nir'].astype(float)
                    ndbi = (swir - nir) / (swir + nir + 1e-8)
                    scene_data['ndbi'] = ndbi.astype(np.float32)
                    indices_count += 1
                
                # NDWI: (NIR - SWIR) / (NIR + SWIR)
                if 'nir' in scene_data and 'b11' in scene_data:
                    nir = scene_data['nir'].astype(float)
                    swir = scene_data['b11'].astype(float)
                    ndwi = (nir - swir) / (nir + swir + 1e-8)
                    scene_data['ndwi'] = ndwi.astype(np.float32)
                    indices_count += 1
                
                # EVI: Enhanced Vegetation Index
                if 'nir' in scene_data and 'red' in scene_data and 'blue' in scene_data:
                    nir = scene_data['nir'].astype(float)
                    red = scene_data['red'].astype(float)
                    blue = scene_data['blue'].astype(float)
                    evi = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1)
                    scene_data['evi'] = evi.astype(np.float32)
                    indices_count += 1
                    
            except Exception as e:
                pass
        
        print(f"✅ Calculated {indices_count} spectral indices per scene")
        
        # ===== STACK SCENES ALONG TIME DIMENSION =====
        print(f"\n[STACKING] Stacking all {len(all_data_dict)} scenes to create time-series...")
        
        data_vars = {}
        band_names = list(all_data_dict[0].keys())
        
        for band_name in band_names:
            band_data_list = []
            for scene_idx in sorted(all_data_dict.keys()):
                if band_name in all_data_dict[scene_idx]:
                    band_data_list.append(all_data_dict[scene_idx][band_name])
            
            if band_data_list:
                stacked = np.stack(band_data_list, axis=0)
                data_vars[band_name] = (['time', 'y', 'x'], stacked)
        
        # Create xarray Dataset with time dimension
        first_band_data = list(all_data_dict[0].values())[0]
        y_size, x_size = first_band_data.shape
        
        data = xr.Dataset(
            data_vars,
            coords={
                'time': np.arange(len(all_data_dict)),
                'x': np.arange(x_size),
                'y': np.arange(y_size)
            }
        )
        
        # ===== CALCULATE TEMPORAL FEATURES FOR ACCURACY =====
        print(f"\n[TEMPORAL FEATURES] Computing temporal features from time-series...")
        
        temporal_features_added = 0
        
        # For NDVI: temporal statistics
        if 'ndvi' in data.data_vars:
            ndvi_ts = data['ndvi']
            
            # Min NDVI (vegetation stress indicator)
            data['ndvi_min'] = ndvi_ts.min(dim='time')
            temporal_features_added += 1
            
            # Max NDVI (peak vegetation)
            data['ndvi_max'] = ndvi_ts.max(dim='time')
            temporal_features_added += 1
            
            # Mean NDVI
            data['ndvi_mean'] = ndvi_ts.mean(dim='time')
            temporal_features_added += 1
            
            # NDVI range (variability)
            data['ndvi_range'] = data['ndvi_max'] - data['ndvi_min']
            temporal_features_added += 1
            
            # NDVI std (temporal consistency)
            data['ndvi_std'] = ndvi_ts.std(dim='time')
            temporal_features_added += 1
        
        # For all indices: mean values (aggregate features)
        for band_name in ['ndbi', 'ndwi', 'evi']:
            if band_name in data.data_vars:
                band_ts = data[band_name]
                data[f'{band_name}_mean'] = band_ts.mean(dim='time')
                temporal_features_added += 1
        
        print(f"✅ Added {temporal_features_added} temporal/aggregate features")
        
        # ===== SAVE TO CACHE =====
        print(f"\n[CACHE] Saving dataset to cache...")
        try:
            data.to_netcdf(cache_file, engine='netcdf4')
            cache_size = os.path.getsize(cache_file) / (1024**3)
            print(f"✅ Dataset saved to cache: {cache_file}")
            print(f"   Cache size: {cache_size:.2f} GB")
        except Exception as e:
            print(f"⚠️  Error saving cache: {e}")
        
        print(f"\n✅ OPTIMAL Dataset with native resolution + temporal features created!")
        print(f"   {'='*70}")
        print(f"   🎬 Total scenes (time steps): {len(all_data_dict)}")
        print(f"   📊 Total bands/variables: {len(data.data_vars)}")
        print(f"   🖼️  Spatial size: {x_size} × {y_size} pixels (NATIVE resolution)")
        print(f"   📏 Native resolution: 10m (Sentinel-2 L2A)")
        print(f"   ⏰ Temporal range: {scene_dates[0].date()} to {scene_dates[-1].date()}")
        print(f"   💾 Total dataset size: {notebook_utils.xarray_object_size(data)}")
        print(f"   💿 Cached at: {cache_file}")
        print(f"   {'='*70}")
        
        print(f"\n   Dataset dimensions:")
        for dim, size in data.dims.items():
            print(f"     {dim}: {size}")
        
        print(f"\n   Variables ({len(data.data_vars)}):")
        spatial_vars = []
        temporal_vars = []
        for var_name in sorted(data.data_vars):
            if len(data[var_name].shape) == 3:
                spatial_vars.append(f"{var_name} {data[var_name].shape}")
            else:
                temporal_vars.append(f"{var_name} {data[var_name].shape}")
        
        print(f"   Spatial time-series ({len(spatial_vars)}):")
        for v in spatial_vars:
            print(f"     - {v}")
        print(f"   Temporal aggregates ({len(temporal_vars)}):")
        for v in temporal_vars:
            print(f"     - {v}")
        
        print(f"   {'='*70}")
        
        display(data)
    
    # ===== EXTRACT NDVI FOR TRAINING =====
    print(f"\n[NDVI EXTRACTION] Extracting NDVI for model training...")
    if 'ndvi_mean' in data.data_vars:
        # Use mean NDVI across time
        ndvi = data['ndvi_mean']
        print(f"✅ NDVI extracted (mean across time)")
        print(f"   Shape: {ndvi.shape}")
    elif 'ndvi' in data.data_vars:
        # Use first time step if mean not available
        ndvi = data['ndvi'].isel(time=0)
        print(f"✅ NDVI extracted (first time step)")
        print(f"   Shape: {ndvi.shape}")
    else:
        print(f"❌ NDVI not found in dataset")
        ndvi = None
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()
    data = None
    ndvi = None

print("="*70)

LOADING SENTINEL-2 DATA FROM S3 COGs (RASTERIO) - OPTIMAL ACCURACY

📥 Downloading from S3...

📦 Found 40 available scenes
   Date range: 2023-03-01 to 2023-12-31

[LOADING] Loading ALL 40 scenes with ALL available bands...
   (Keeping NATIVE resolution - NO upsampling/magnification)
   Available bands: ['nir', 'red', 'scl', 'blue', 'green', 'nir08', 'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2', 'rededge3']

   [ 1/1] S2A_48PWR_20231226_0_L2A (2023-12-26)
      ✅ 13 bands loaded

✅ Successfully loaded 1 scenes!

[RESOLUTION NORMALIZATION] Aligning all bands to native resolution (NO magnification)...
   Reference resolution: 10980×10980 pixels (native nir)
      Resampling scl: 5490×5490 → 10980×10980
      Resampling nir08: 5490×5490 → 10980×10980
      Resampling nir09: 1830×1830 → 10980×10980
      Resampling swir16: 5490×5490 → 10980×10980
      Resampling swir22: 5490×5490 → 10980×10980
      Resampling coastal: 1830×1830 → 10980×10980
      Resampling rededge1: 54

<timed exec>:169: RuntimeWarning: divide by zero encountered in divide
<timed exec>:169: RuntimeWarning: invalid value encountered in divide


✅ Calculated 2 spectral indices per scene

[STACKING] Stacking all 1 scenes to create time-series...

[TEMPORAL FEATURES] Computing temporal features from time-series...
✅ Added 6 temporal/aggregate features

[CACHE] Saving dataset to cache...
✅ Dataset saved to cache: dataset_cache/sentinel2_timeseries_40scenes.nc
   Cache size: 6.40 GB

✅ OPTIMAL Dataset with native resolution + temporal features created!
   🎬 Total scenes (time steps): 1
   📊 Total bands/variables: 21
   🖼️  Spatial size: 10980 × 10980 pixels (NATIVE resolution)
   📏 Native resolution: 10m (Sentinel-2 L2A)
   ⏰ Temporal range: 2023-12-26 to 2023-12-26
   💾 Total dataset size: Dataset size: 6.40 GB
   💿 Cached at: dataset_cache/sentinel2_timeseries_40scenes.nc

   Dataset dimensions:
     time: 1
     y: 10980
     x: 10980

   Variables (21):
   Spatial time-series (15):
     - blue (1, 10980, 10980)
     - coastal (1, 10980, 10980)
     - evi (1, 10980, 10980)
     - green (1, 10980, 10980)
     - ndvi (1, 10980, 1

<timed exec>:267: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


<xarray.Dataset> Size: 7GB
Dimensions:     (time: 1, y: 10980, x: 10980)
Coordinates:
  * time        (time) int64 8B 0
  * x           (x) int64 88kB 0 1 2 3 4 5 ... 10975 10976 10977 10978 10979
  * y           (y) int64 88kB 0 1 2 3 4 5 ... 10975 10976 10977 10978 10979
Data variables: (12/21)
    nir         (time, y, x) uint16 241MB 322 336 337 342 ... 384 362 369 400
    red         (time, y, x) uint16 241MB 769 783 792 796 ... 722 701 658 747
    scl         (time, y, x) uint8 121MB 6 6 6 6 6 6 6 6 6 ... 6 6 6 6 6 6 6 6 6
    blue        (time, y, x) uint16 241MB 522 516 524 527 ... 1001 1014 1000
    green       (time, y, x) uint16 241MB 718 728 751 748 ... 1160 1116 1152
    nir08       (time, y, x) uint16 241MB 281 278 275 280 ... 437 411 443 474
    ...          ...
    ndvi_min    (y, x) float32 482MB -0.4097 -0.3995 -0.403 ... -0.2814 -0.3025
    ndvi_max    (y, x) float32 482MB -0.4097 -0.3995 -0.403 ... -0.2814 -0.3025
    ndvi_mean   (y, x) float32 482MB -0.4097 -0.3995 -0.403 ... -0.2814 -0.3025
    ndvi_range  (y, x) float32 482MB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    ndvi_std    (y, x) float32 482MB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    evi_mean    (y, x) float32 482MB -1.093 -0.9592 -0.9806 ... 0.2198 0.3315

CPU times: user 1min 9s, sys: 32 s, total: 1min 41s
Wall time: 3min 50s


In [13]:
# 🎯 LOAD TRAINING DATA & EXTRACT FEATURES
print("="*70)
print("TRAINING DATA SETUP")
print("="*70)

# Load training points
train_path = "train/ST_training data_updated_1130points_new.shp"
print(f"\n[1] Loading training data: {train_path}")

try:
    train = load_train_data(train_path)
    print(f"    ✅ Loaded {len(train)} training points")
    print(f"    Columns: {list(train.columns)}")
    train.head()
except Exception as e:
    print(f"    ❌ Error: {e}")
    train = None

# Label mapping
label_mapping = {
    "Lua tom": "0",
    "Lua": "1",
    "CHN": "2",
    "CLN": "3",
    "TS": "4",
    "Song": "5",
    "Dat xay dung": "6",
    "Rung": "7",
}

print(f"\n[2] Label mapping:")
for label, code in label_mapping.items():
    print(f"    {code}: {label}")

print("\n" + "="*70)

TRAINING DATA SETUP

[1] Loading training data: train/ST_training data_updated_1130points_new.shp
    ✅ Loaded 1130 training points
    Columns: ['No', 'X', 'Y', 'LU2022', 'Hientrang', 'HT_code', 'geometry']

[2] Label mapping:
    0: Lua tom
    1: Lua
    2: CHN
    3: CLN
    4: TS
    5: Song
    6: Dat xay dung
    7: Rung



In [11]:
%%time
# 🤖 RANDOM FOREST MODEL TRAINING
print("="*70)
print("MODEL TRAINING")
print("="*70)

if train is not None and ndvi is not None:
    print("\n[1] Extracting features from NDVI...")
    try:
        # Extract NDVI values at training point locations
        X = []
        y = []
        
        for idx, point in train.iterrows():
            try:
                # Get NDVI value at point location (nearest neighbor)
                ndvi_val = float(ndvi.sel(x=point.geometry.x, y=point.geometry.y, method='nearest').values)
                label = label_mapping[point.Hientrang]
                
                X.append([ndvi_val])
                y.append(int(label))
            except Exception as e:
                print(f"    ⚠️  Point {idx}: {e}")
        
        if len(X) > 0:
            X = np.array(X)
            y = np.array(y)
            print(f"    ✅ Extracted {len(X)} samples")
            
            # Split data
            print(f"\n[2] Splitting data (80-20)...")
            from sklearn.model_selection import train_test_split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )
            print(f"    Train: {len(X_train)}, Test: {len(X_test)}")
            
            # Train model
            print(f"\n[3] Training Random Forest...")
            from sklearn.ensemble import RandomForestClassifier
            from sklearn.metrics import accuracy_score
            
            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            model.fit(X_train, y_train)
            
            # Evaluate
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            print(f"    ✅ Model trained!")
            print(f"    Accuracy: {accuracy*100:.2f}%")
            
        else:
            print(f"    ❌ No samples extracted")
            model = None
            
    except Exception as e:
        print(f"    ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        model = None
else:
    print("❌ Missing training data or NDVI")
    model = None

print("="*70)

MODEL TRAINING

[1] Extracting features from NDVI...
    ⚠️  Point 0: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 1: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 2: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 3: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 4: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 5: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 6: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 7: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 8: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 9: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 10: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 11: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 12: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 13: 'numpy.ndarray' object has no attribute 'sel'
    ⚠️  Point 14: 'numpy.ndarray' object has no attribute 'se

In [12]:
# 💾 SAVE MODEL
print("="*70)
print("MODEL SAVING")
print("="*70)

if model is not None:
    print("\n🔄 Saving trained model...")
    try:
        save_model("model_rasterio.joblib", model)
        print("✅ Model saved to model_train/model_rasterio.joblib")
    except Exception as e:
        print(f"❌ Error saving model: {e}")
else:
    print("❌ No model to save")

print("="*70)

MODEL SAVING
❌ No model to save


In [ ]:
# 🛑 CLEANUP
print("="*70)
print("CLEANUP")
print("="*70)

print("\n🔄 Closing Dask client and cluster...")
try:
    client.close()
    cluster.close()
    print("✅ Cleanup complete")
except Exception as e:
    print(f"⚠️  Error during cleanup: {e}")

print("\n" + "="*70)
print("✅ PIPELINE COMPLETE")
print("="*70)

CLEANUP

🔄 Closing Dask client and cluster...
✅ Cleanup complete

✅ PIPELINE COMPLETE
✅ Cleanup complete

✅ PIPELINE COMPLETE
